# 02 — اختبار مكونات الـ Pipeline (Unit-level)

**الهدف:** اختبار كل مكوّن لوحده قبل ما نختبرهم مع بعض في notebook 03: **Extraction → Chunking → Embedding**، وكمان الـ Python parser.

**ملاحظة مهمة:** أنا مش شايف الكود بتاعك، فالـ notebook بيعمل *auto-discovery* للـ classes/functions جوه `pipeline/` ويطبعها في جدول. لو اختار حاجة غلط، عدّل خلية **ADAPT** بإيدك. أي اختبار مش قادر يشتغل بيتعلّم `SKIP` مع سبب واضح (مش بيفشل بشكل مضلل).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("02_pipeline_components")

In [ ]:
import pkgutil, importlib, inspect, tempfile
import numpy as np
import matplotlib.pyplot as plt

corpus = make_corpus()
rows, import_errors = [], []
try:
    import pipeline
    for m in pkgutil.walk_packages(pipeline.__path__, "pipeline."):
        try:
            mod = importlib.import_module(m.name)
        except Exception as e:
            import_errors.append((m.name, f"{type(e).__name__}: {e}"))
            continue
        for n, o in inspect.getmembers(mod):
            if n.startswith("_") or getattr(o, "__module__", None) != m.name:
                continue
            if inspect.isfunction(o) or inspect.isclass(o):
                try:
                    sig = str(inspect.signature(o))
                except Exception:
                    sig = "?"
                rows.append(dict(module=m.name, kind="class" if inspect.isclass(o) else "function", name=n, signature=sig[:110]))
except ImportError as e:
    print("pipeline package not importable:", e)
display(pd.DataFrame(rows))
rec.check("all pipeline modules import cleanly", not import_errors, str(import_errors)[:300], "pipeline")

## ADAPT — ربط المكوّنات
اتأكد من الجدول اللي فوق. لو الـ auto-discovery غلط، اكتب الـ import بتاعك مباشرة، مثال:
`from pipeline.chunking.recursive_chunker import RecursiveChunker; CHUNKER = RecursiveChunker`

In [ ]:
CHUNKER   = find_callable("pipeline.chunking", "chunk", "split")
EXTRACTOR = find_callable("pipeline.extraction", "extract", "load", "read", "parse")
PY_PARSER = find_callable("pipeline", "python", "pycode", "ast")
EMBEDDER  = find_callable("pipeline.embedding", "embed") or find_callable("helpers.embeddings", "embed")
display(pd.DataFrame([(k, getattr(v, "__module__", "") + "." + getattr(v, "__name__", "") if v else "— not found —")
                      for k, v in dict(CHUNKER=CHUNKER, EXTRACTOR=EXTRACTOR, PY_PARSER=PY_PARSER, EMBEDDER=EMBEDDER).items()],
                     columns=["adapter", "resolved to"]))

def _invoke(obj, method_names, *args, **kw):
    if obj is None:
        raise Skip("component not found - set it in the ADAPT cell")
    target = obj
    if inspect.isclass(obj):
        try:
            target = obj()
        except TypeError:
            raise Skip(f"{obj.__name__}() needs constructor args - instantiate it manually in the ADAPT cell")
        for m in method_names:
            if hasattr(target, m):
                return call_flex(getattr(target, m), *args, **kw)
        raise Skip(f"{obj.__name__} has none of the methods {method_names}")
    try:
        return call_flex(target, *args, **kw)
    except TypeError as e:
        raise Skip(f"signature mismatch ({e}) - adapt the wrapper functions below")

def norm_chunks(res):
    out = []
    for x in res or []:
        if isinstance(x, str):
            out.append(x)
        elif isinstance(x, dict):
            out.append(next((x[k] for k in ("text", "content", "chunk_text", "page_content") if k in x), ""))
        else:
            out.append(next((getattr(x, k) for k in ("text", "content", "chunk_text", "page_content") if hasattr(x, k)), str(x)))
    return out

def run_chunker(text, **kw):
    return norm_chunks(_invoke(CHUNKER, ("chunk", "split", "split_text", "chunk_text", "chunk_document", "run", "__call__"), text, **kw))

def run_extractor(path):
    res = _invoke(EXTRACTOR, ("extract", "extract_text", "load", "read", "parse", "run", "__call__"), str(path))
    return "\n".join(flatten_strings(res)) if not isinstance(res, str) else res

def run_py_parser(src_or_path):
    res = _invoke(PY_PARSER, ("parse", "extract", "run", "__call__"), src_or_path)
    return "\n".join(flatten_strings(res)) if not isinstance(res, str) else res

## 1) Embedding model (الموديل الخام)
بنحمّل الموديل مرة واحدة ونستخدمه في اختبار الـ chunk lengths (الـ tokens) واختبارات الـ semantic.

In [ ]:
M = None
with rec.case("load embedding model", "embedding") as c:
    from sentence_transformers import SentenceTransformer
    M = SentenceTransformer(CONFIG["EMBEDDING_MODEL"])
    c.note(f"max_seq_length={M.max_seq_length}")

def emb(texts, kind="passage"):
    return M.encode([f"{kind}: {t}" for t in texts], normalize_embeddings=True)

## 2) Extraction
لكل نوع ملف: هل النص المستخرج فيه المعلومة المتوقعة؟ وهل العربي سليم؟

In [ ]:
tmp = pathlib.Path(tempfile.mkdtemp(prefix="rag_nb_"))
EXTRACTED = {}
expect = {"orion_manual.txt": "450 degrees", "blue_nile_plan_ar.txt": "أحمد فؤاد",
          "inventory.py": "calculate_reorder_point", "operations_report.pdf": "3.7 percent"}
for name, (data, mime) in corpus.items():
    f = tmp / name
    f.write_bytes(data)
    with rec.case(f"extract text: {name}", "extraction") as c:
        txt = run_extractor(f)
        EXTRACTED[name] = txt
        assert txt and txt.strip(), "empty extraction"
        assert expect[name].lower() in txt.lower(), f"expected fragment '{expect[name]}' missing"
        c.note(f"{len(txt)} chars")

with rec.case("Arabic text is not corrupted (no mojibake / reversed letters)", "extraction") as c:
    need(EXTRACTED.get("blue_nile_plan_ar.txt"))
    txt = EXTRACTED["blue_nile_plan_ar.txt"]
    src = corpus["blue_nile_plan_ar.txt"][0].decode("utf-8")
    ar_src = set(re.findall(r"[\u0600-\u06FF]", src))
    ar_out = set(re.findall(r"[\u0600-\u06FF]", txt))
    assert ar_src <= ar_out, f"missing Arabic letters: {''.join(sorted(ar_src - ar_out))}"
    assert "\ufffd" not in txt, "replacement characters found (encoding problem)"

with rec.case("PDF keeps page information (both pages extracted)", "extraction") as c:
    need(EXTRACTED.get("operations_report.pdf"))
    t = EXTRACTED["operations_report.pdf"].lower()
    assert "regional operations" in t and "churn" in t, "one of the PDF pages is missing"

In [ ]:
def outcome(fn):
    try:
        r = fn()
        return f"returned {repr(r)[:60]}"
    except Skip:
        raise
    except Exception as e:
        return f"raised {type(e).__name__}: {str(e)[:60]}"

with rec.case("edge case: empty .txt file (documented behaviour)", "extraction") as c:
    f = tmp / "empty.txt"; f.write_bytes(b"")
    c.note(outcome(lambda: run_extractor(f)))
with rec.case("edge case: corrupted PDF (must not hang or crash the worker)", "extraction") as c:
    f = tmp / "broken.pdf"; f.write_bytes(b"%PDF-1.4\nthis is not a real pdf")
    c.note(outcome(lambda: run_extractor(f)))
with rec.case("edge case: unsupported extension .xyz", "extraction") as c:
    f = tmp / "file.xyz"; f.write_bytes(b"hello")
    c.note(outcome(lambda: run_extractor(f)))

## 3) Chunking
أهم خصائص الـ chunker: ما يطلعش chunks فاضية، الحجم محدود، مفيش نص بيضيع، الترتيب محفوظ، فيه overlap، deterministic، وكمان **الـ chunk ما يعديش حد الـ tokens بتاع الموديل** (وإلا الموديل بيقص الباقي بصمت).

In [ ]:
words = lambda t: set(re.findall(r"\w+", t.lower()))
en = corpus["orion_manual.txt"][0].decode("utf-8")
ar = corpus["blue_nile_plan_ar.txt"][0].decode("utf-8")
CH = {}
for label, txt in (("english", en), ("arabic", ar)):
    with rec.case(f"chunker returns chunks ({label})", "chunking") as c:
        CH[label] = run_chunker(txt)
        assert CH[label], "no chunks returned"
        c.note(f"{len(CH[label])} chunks, avg {np.mean([len(x) for x in CH[label]]):.0f} chars")

for label, txt in (("english", en), ("arabic", ar)):
    ch = CH.get(label)
    with rec.case(f"no empty / whitespace-only chunks ({label})", "chunking"):
        need(ch); assert all(x.strip() for x in ch), "found empty chunks"
    with rec.case(f"max chunk size <= {CONFIG['MAX_CHUNK_CHARS']} chars ({label})", "chunking") as c:
        need(ch); mx = max(len(x) for x in ch); assert mx <= CONFIG["MAX_CHUNK_CHARS"], f"largest chunk = {mx} chars"
    with rec.case(f"no content lost: every word of the source appears in chunks ({label})", "chunking") as c:
        need(ch); lost = words(txt) - words(" ".join(ch)); assert not lost, f"lost words: {sorted(lost)[:10]}"
    with rec.case(f"order preserved: first/last chunk match document start/end ({label})", "chunking"):
        need(ch)
        assert txt.strip().split()[0].lower() in ch[0].lower(), "first chunk does not contain the first word"
        assert txt.strip().split()[-1].lower() in ch[-1].lower(), "last chunk does not contain the last word"
    with rec.case(f"consecutive chunks overlap ({label})", "chunking") as c:
        need(ch)
        if len(ch) < 2:
            raise Skip("only one chunk")
        ov = sum(1 for a, b in zip(ch, ch[1:]) if words(a) & words(b) and any(w in b for w in a.split()[-5:]))
        if ov == 0:
            raise Warn("no overlap between chunks: an answer spanning two chunks can be missed")
        c.note(f"{ov}/{len(ch) - 1} pairs overlap")
    with rec.case(f"deterministic: same input -> same chunks ({label})", "chunking"):
        need(ch); assert run_chunker(txt) == ch, "two runs gave different chunks"

with rec.case("tiny input (one sentence) -> exactly one chunk", "chunking"):
    r = run_chunker("This is a single short sentence.")
    assert len(r) == 1, f"got {len(r)} chunks"
with rec.case("empty / whitespace input does not crash", "chunking") as c:
    c.note(outcome(lambda: run_chunker("   \n  ")))
with rec.case("Arabic letters survive chunking", "chunking"):
    need(CH.get("arabic")); assert set(re.findall(r"[\u0600-\u06FF]", ar)) <= set(re.findall(r"[\u0600-\u06FF]", " ".join(CH["arabic"])))

In [ ]:
with rec.case("chunks fit inside the model's token window (no silent truncation)", "chunking") as c:
    need(M, CH.get("english"), CH.get("arabic"))
    lens = [len(M.tokenizer.encode("passage: " + x, add_special_tokens=True)) for x in CH["english"] + CH["arabic"]]
    limit = M.max_seq_length
    c.note(f"max tokens={max(lens)} / model limit={limit}")
    if max(lens) > limit:
        raise AssertionError(f"{sum(l > limit for l in lens)} chunks exceed {limit} tokens -> the tail of those chunks is NOT embedded")

if CH:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    for a, (k, v) in zip(ax, CH.items()):
        a.bar(range(len(v)), [len(x) for x in v]); a.set_title(f"chunk sizes (chars) - {k}"); a.set_xlabel("chunk #")
    plt.tight_layout(); plt.show()

## 4) Python parser
(الخطوة 2 في الـ roadmap — لو لسه ما اتعملتش هتلاقي SKIP)

In [ ]:
with rec.case("python file -> class & function names are captured", "python-parser") as c:
    f = tmp / "inventory.py"
    out = run_py_parser(str(f))
    for sym in ("InventoryManager", "calculate_reorder_point", "format_report"):
        assert sym in out, f"symbol '{sym}' missing from parser output"
    assert "reorder point" in out.lower(), "docstring text missing"
with rec.case("syntax-error file does not crash the parser (falls back or fails cleanly)", "python-parser") as c:
    bad = tmp / "broken.py"; bad.write_text("def broken(:\n    pass\n")
    c.note(outcome(lambda: run_py_parser(str(bad))))

## 5) جودة الـ Embeddings (semantic)
مهم تفتكر: e5 بيدّي cosine عالي حتى للنصوص الغير مرتبطة، فبنقارن **ترتيب** النتايج مش القيمة المطلقة.

In [ ]:
with rec.case(f"embedding shape = (n, {CONFIG['EMBED_DIM']}), finite values", "embedding") as c:
    need(M); v = emb(["a", "b", "c"]); assert v.shape == (3, CONFIG["EMBED_DIM"]); assert np.isfinite(v).all(), "NaN/inf in embeddings"
with rec.case("deterministic (same text -> same vector)", "embedding"):
    need(M); a, b = emb(["same text"]), emb(["same text"]); assert np.allclose(a, b, atol=1e-5)
with rec.case("batch result == single result (padding does not change vectors)", "embedding") as c:
    need(M)
    texts = [CH["english"][0], "short", CH["arabic"][0]] if CH.get("english") and CH.get("arabic") else ["short one", "a much longer sentence " * 20]
    batch = emb(texts); single = np.vstack([emb([t]) for t in texts])
    d = float(np.abs(batch - single).max()); c.note(f"max abs diff={d:.5f}"); assert d < 5e-3
with rec.case("empty / whitespace text does not crash the model", "embedding"):
    need(M); v = emb(["", "   "]); assert np.isfinite(v).all()

def rank_test(query, passages, correct_idx):
    q = emb([query], "query"); p = emb(passages, "passage"); s = (q @ p.T)[0]
    return int(np.argmax(s)), s

TESTS = [
    ("English retrieval", "What is the capital of France?",
     ["Paris is the capital and largest city of France.", "Bananas are rich in potassium.", "The stock market closed higher on Friday."], 0),
    ("Arabic retrieval", "ما هي عاصمة مصر؟",
     ["يعتبر الأرز من المحاصيل الغذائية الأساسية.", "القاهرة هي عاصمة جمهورية مصر العربية.", "تقام مباراة كرة القدم يوم الجمعة."], 1),
    ("Arabic query -> English passage", "ما هي عاصمة فرنسا؟",
     ["Bananas are rich in potassium.", "Paris is the capital and largest city of France.", "The stock market closed higher on Friday."], 1),
    ("English query -> Arabic passage", "What is the capital of Egypt?",
     ["الأرز من المحاصيل الغذائية الأساسية.", "تقام المباراة يوم الجمعة.", "القاهرة هي عاصمة جمهورية مصر العربية."], 2),
]
sem_rows = []
for name, q, ps, ok in TESTS:
    with rec.case(f"semantic ranking: {name}", "embedding") as c:
        need(M); top, s = rank_test(q, ps, ok)
        sem_rows.append(dict(test=name, scores=[round(float(x), 3) for x in s], top=top, expected=ok))
        assert top == ok, f"top-1 was passage #{top}, expected #{ok}"
if sem_rows: display(pd.DataFrame(sem_rows))

with rec.case("e5 prefixes matter: 'query:'/'passage:' ranking is correct", "embedding") as c:
    need(M)
    q, ps = TESTS[0][1], TESTS[0][2]
    with_p = (emb([q], "query") @ emb(ps, "passage").T)[0]
    without = (M.encode([q], normalize_embeddings=True) @ M.encode(ps, normalize_embeddings=True).T)[0]
    margin = lambda s: float(s[0] - max(s[1:]))
    c.note(f"margin with prefixes={margin(with_p):.3f}, without={margin(without):.3f}")
    assert np.argmax(with_p) == 0

## 6) الـ Embedder بتاع التطبيق ≡ الموديل الخام + prefixes؟
بنقارن output الـ wrapper بتاعك مع الموديل مباشرة. لو مش متطابقين، يبقى فيه prefix ناقص أو normalization مختلف.

In [ ]:
def call_embedder(kind, texts):
    obj = EMBEDDER
    if obj is None:
        raise Skip("embedder wrapper not found - set EMBEDDER in the ADAPT cell")
    inst = obj() if inspect.isclass(obj) else obj
    names = {"query": ("embed_query", "embed_queries", "encode_query"),
             "passage": ("embed_passages", "embed_documents", "embed_texts", "embed_chunks", "encode_passages", "embed")}[kind]
    for n in names:
        fn = getattr(inst, n, None) if not inspect.isfunction(inst) else None
        if fn:
            res = fn(texts if n != "embed_query" else texts[0])
            return np.asarray(res, dtype="float32").reshape(len(texts), -1)
    raise Skip(f"none of {names} found on the embedder - adapt call_embedder()")

for kind in ("passage", "query"):
    with rec.case(f"app embedder ({kind}) == raw model with '{kind}:' prefix", "embedding") as c:
        need(M)
        texts = ["Orion-7 operates at 450 degrees.", "ميزانية المشروع 4.5 مليون جنيه"]
        mine = call_embedder(kind, texts)
        assert mine.shape == (2, CONFIG["EMBED_DIM"]), f"shape {mine.shape}"
        ref = emb(texts, kind)
        mine_n = mine / np.linalg.norm(mine, axis=1, keepdims=True)
        cos = (mine_n * ref).sum(axis=1)
        c.note(f"cosine to reference={np.round(cos, 4).tolist()}")
        assert (cos > 0.999).all(), "app embeddings differ from model+prefix -> missing/incorrect prefix or wrong model"

## الملخص

In [ ]:
rec.summary(); rec.table(); rec.save()